# 第4章 第一个程序 + 性能分析 - 操作手册

**Goal**: 跑通 vector add kernel，建立 baseline benchmark，理解算术强度和有效带宽。

**Prerequisite**: 已完成第1-3章，理解 GPU 体系结构和内存层级。

**Platform**: 原生 Ubuntu 24.04（推荐）或 WSL2，gfx1201 为叙述基线。

**参考文档**: `docs/part0-intro/chapter4/index.md`

**代码目录**: `code/part0-intro/chapter4/`

## 1. 定位仓库根目录

## 2. 检测 GPU 架构

**Parameter**: 无

**Execution**: 运行 `rocminfo` 检测当前 GPU 架构。

**Expected output**: 输出检测到的架构（gfx1100/gfx1151/gfx1201）。

**Pass criteria**: 成功检测到支持的架构之一。

In [ ]:
# 检测当前 GPU 架构
import subprocess

rocminfo_result = subprocess.run(
    ["rocminfo"],
    capture_output=True,
    text=True,
    check=True
)

# 从 rocminfo 输出中提取架构
arch = "gfx1201"  # 默认值（gfx1201 为叙述基线）
if "gfx1100" in rocminfo_result.stdout:
    arch = "gfx1100"
elif "gfx1151" in rocminfo_result.stdout:
    arch = "gfx1151"
elif "gfx1201" in rocminfo_result.stdout:
    arch = "gfx1201"
else:
    raise RuntimeError("未检测到支持的架构 (gfx1100/gfx1151/gfx1201)")

print(f"检测到架构: {arch}")


In [ ]:
import pathlib
import subprocess

def find_repo_root():
    current = pathlib.Path.cwd().resolve()
    for candidate in [current] + list(current.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").is_dir()
            and (candidate / "docs").is_dir()
            and (candidate / "notebooks").is_dir()
        ):
            return candidate
    raise FileNotFoundError("无法定位仓库根目录")

REPO_ROOT = find_repo_root()
print(f"仓库根目录: {REPO_ROOT}")

## 3. 编译并运行 vector_add

**Parameter**:
- 源文件: `code/part0-intro/chapter4/vector_add.hip`
- 编译选项: `-O2`
- 目标架构: gfx1201（根据实际 GPU 调整）

**Execution**: 编译并运行 vector_add kernel。

**Expected output**: 
- 设备名称
- 向量大小
- blocks 和 threads_per_block
- max_error 为 0
- status: PASS

**Pass criteria**: 编译成功，运行输出 status: PASS。

**Platform-specific commands**:
- gfx1100: `hipcc --offload-arch=gfx1100 -O2 vector_add.hip -o vector_add`
- gfx1151: `hipcc --offload-arch=gfx1151 -O2 vector_add.hip -o vector_add`
- gfx1201: `hipcc --offload-arch=gfx1201 -O2 vector_add.hip -o vector_add`

In [ ]:
chapter4_dir = REPO_ROOT / "code/part0-intro/chapter4"
vector_add_hip = chapter4_dir / "vector_add.hip"
vector_add_bin = chapter4_dir / "vector_add"

# 编译（以 gfx1201 为基线）
compile_result = subprocess.run(
    ["hipcc", f"--offload-arch={arch}", "-O2",
     str(vector_add_hip), "-o", str(vector_add_bin)],
    capture_output=True,
    text=True,
    check=True,
    cwd=chapter4_dir
)
print("编译成功")

# 运行
run_result = subprocess.run(
    [str(vector_add_bin)],
    capture_output=True,
    text=True,
    check=True,
    cwd=chapter4_dir
)
print("\n运行结果:")
print(run_result.stdout)

if "status: PASS" in run_result.stdout:
    print("\n✓ Vector Add 验证通过")
else:
    print("\n✗ Vector Add 验证失败")

## 4. Baseline Benchmark

**Parameter**:
- 向量大小: 16M 元素（2^24）
- warmup: 5 次
- repeat: 30 次

**Execution**: 运行 `benchmark_vector_add.py`，测量 CPU 和 GPU 的性能。

**Expected output**:
- CPU 和 GPU 的 mean/median/min 执行时间
- GPU 的有效带宽（基于 min 时间）

**Pass criteria**: 
- 程序成功运行
- GPU 性能显著优于 CPU
- 输出包含 warmup 和 repeat 参数

In [ ]:
benchmark_script = chapter4_dir / "benchmark_vector_add.py"

# 运行 benchmark
run_result = subprocess.run(
    ["python", str(benchmark_script),
     "--size", "16777216",  # 2^24
     "--warmup", "5",
     "--repeat", "30"],
    capture_output=True,
    text=True,
    check=True,
    cwd=chapter4_dir
)
print("Benchmark 结果:")
print(run_result.stdout)

print("\n关键指标说明:")
print("- warmup: 预热次数，让 GPU 进入稳定状态")
print("- repeat: 重复测量次数，用于统计分析")
print("- min_ms: 最小执行时间，更接近无干扰的真实性能")
print("- bandwidth_gb_s: 有效带宽 = (3 × N × 4 bytes) / time")

## 5. 算术强度和有效带宽分析

**Parameter**: 无

**Execution**: 计算 vector add 的算术强度和有效带宽。

**Expected output**: 
- 算术强度（FLOP/Byte）
- 逻辑带宽（基于算法字节数）
- 物理带宽（需要硬件计数器，此处仅说明概念）

**Pass criteria**: 正确计算并解释算术强度和带宽概念。

In [ ]:
# 算术强度和带宽分析
N = 16777216  # 2^24 元素
bytes_per_element = 4  # FP32

# Vector Add: c[i] = a[i] + b[i]
# 读取: a[i], b[i] (2 reads)
# 写入: c[i] (1 write)
# 计算: 1 次加法

reads = 2 * N * bytes_per_element
writes = 1 * N * bytes_per_element
total_bytes = reads + writes
flops = N  # 1 次加法 per 元素

arithmetic_intensity = flops / total_bytes

print("算术强度分析:")
print(f"- 总元素数: {N:,}")
print(f"- 读取字节: {reads:,} bytes ({reads / 1e9:.3f} GB)")
print(f"- 写入字节: {writes:,} bytes ({writes / 1e9:.3f} GB)")
print(f"- 总字节数: {total_bytes:,} bytes ({total_bytes / 1e9:.3f} GB)")
print(f"- 浮点运算: {flops:,} FLOPs")
print(f"- 算术强度: {arithmetic_intensity:.6f} FLOP/Byte")

print("\n带宽概念:")
print("- 逻辑带宽: 基于算法字节数（3 × N × 4）计算的带宽")
print("- 物理带宽: 实际在 GDDR6 上传输的字节数（需硬件计数器）")
print("- 有效带宽 (effective bandwidth): 逻辑字节数 / 实测时间")
print("- 理论峰值: 必须从当前平台官方规格或项目实测基线中注明来源后填写")

print("\n解释:")
print("- Vector Add 是典型的访存密集型算子（arithmetic intensity 很低）")
print("- 性能瓶颈在内存带宽，而非计算能力")
print("- 优化重点是提高内存访问效率（合并访存、缓存利用）")

## 6. Warmup 和 Repeat 的重要性

**Parameter**: 无

**Execution**: 说明 warmup 和 repeat 在性能测试中的作用。

**Expected output**: 概念解释和最佳实践。

**Pass criteria**: 正确理解 warmup 和 repeat 的必要性。

In [ ]:
print("Warmup 的作用:")
print("- GPU 初始状态可能不稳定（频率调整、缓存冷启动）")
print("- Warmup 让 GPU 进入稳定工作状态")
print("- 通常 5-10 次 warmup 足够")

print("\nRepeat 的作用:")
print("- 单次测量容易受外部干扰（系统调度、其他进程）")
print("- 多次重复测量可以统计分析（mean/median/min）")
print("- Min 更接近无干扰的真实性能")
print("- 通常 20-50 次 repeat 可以得到稳定结果")

print("\n最佳实践:")
print("- 正式计时前先 warmup")
print("- 使用 GPU Event 或 synchronize 确保计时准确")
print("- 报告 min/median/mean，不只报告单次结果")
print("- 记录测试参数（size, warmup, repeat）以便复现")

## 总结

本章完成了第一个完整的 GPU 程序和性能分析：
1. ✓ 跑通 vector add kernel
2. ✓ 建立 baseline benchmark（warmup + repeat）
3. ✓ 理解算术强度和有效带宽
4. ✓ 区分逻辑带宽和物理带宽
5. ✓ 掌握性能测试的最佳实践

这些是后续 profiling 和优化工作的基础。